# H03 — Correlation and the power spectral density

**Part IV · random processes, their correlation and their spectrum**

> **References**
>
> (Therrien 1992), (Papoulis and Pillai 2002), (Kay 1988), (Harris 1978)

## What you do here

The noise of a detector is a random process. We cannot say what the next sample will be but we can say how the samples behave on average. In this laboratory you meet the random process and its realisations. You measure how the samples of one record are linked: the autocorrelation. You measure where the power is: the power spectral density. You see that the two are the same information. Then you estimate the spectrum from one record, with a window and with segments.

The figure below is the map of the whole laboratory. The top row is what the process is. The bottom row is how we estimate it from the record we have. Tasks 1 to 5 fill the top row and tasks 6 and 7 the bottom row.

All the code is given. In every task you change the parameters on the lines under `# change these`, run the cell again and answer one question.

> **The notation**
>
> - **$n$** — the step in time, an integer
> - **$x^{(i)}[n]$** — the sample at step $n$ in the realisation $i = 1, \ldots, R$ of the process
> - **$R$** — the number of realisations
> - **$N$** — the number of samples in one record
> - **$k$** — the lag: the distance in steps between two samples
> - **$f$** — the frequency in cycles per sample, $-\tfrac{1}{2} \le f \le \tfrac{1}{2}$
> - **$\mathcal{E}\{\cdot\}$** — the expectation: the average over the realisations
> - **$r_{xx}[k]$, $S_{xx}(f)$** — the autocorrelation and the power spectral density
> - **$H[n]$** — the window
> - **$D$** — the number of samples in one segment of the record

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

> **How to use this notebook**
>
> Run the cells in order. Each task has the same parts. **What we do** says the physics. The code cells define the functions of the task and then call them. The random numbers are seeded, so a run gives the numbers of the solutions. **What to change** says which parameters to vary and what to look for. Answer the question before you open the solution. Every function is defined in the task where it is first used. A later task says which earlier functions it uses.

### The workflow drawing

The function `draw_flow` draws the maps of this notebook: boxes joined by arrows. You do not need to read it. It is used here and in tasks 5 and 7.

In [ ]:
def draw_flow(boxes, arrows, size=(12, 4)):
    """Draw a workflow: boxes joined by arrows.

    boxes  : list of (x, y, width, text, colour), the centre of the box in
             drawing units, its width in the same units, its text and the
             name of a palette colour, for example 'BLUE'
    arrows : list of (i, j, kind, label), an arrow from box i to box j;
             kind is 'to', 'both' or 'dashed'
    size   : width and height of the figure in inches
    Returns nothing. The figure is shown."""
    fig, ax = plt.subplots(figsize=size)
    height = 1.0
    patches = []
    # one rounded box with its text for every step of the workflow
    for (x, y, width, text, colour) in boxes:
        patch = FancyBboxPatch((x - width / 2, y - height / 2), width, height,
                               boxstyle='round,pad=0.02,rounding_size=0.12',
                               facecolor=P[colour + '_PALE'], edgecolor=P[colour], lw=1.6)
        ax.add_patch(patch)
        ax.text(x, y, text, ha='center', va='center', fontsize=10.5, color=P['INK'])
        patches.append(patch)
    # one arrow for every link; a label sits beside its arrow
    for (i, j, kind, label) in arrows:
        xi, yi = boxes[i][0], boxes[i][1]
        xj, yj = boxes[j][0], boxes[j][1]
        style = '<->' if kind == 'both' else '->'
        line = '--' if kind == 'dashed' else '-'
        ax.annotate('', xy=(xj, yj), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle=style, color=P['GREY_DARK'], lw=1.6, ls=line,
                                    patchA=patches[i], patchB=patches[j],
                                    shrinkA=2, shrinkB=2, mutation_scale=16))
        if label:
            vertical = abs(xj - xi) < abs(yj - yi)
            ax.text((xi + xj) / 2 + (0.15 if vertical else 0), (yi + yj) / 2 + (0 if vertical else 0.12),
                    label, ha='left' if vertical else 'center', va='center' if vertical else 'bottom',
                    fontsize=9.5, color=P['GREY_DARK'])
    # the limits follow the boxes
    ax.set_xlim(min(b[0] - b[2] / 2 for b in boxes) - 0.2, max(b[0] + b[2] / 2 for b in boxes) + 0.2)
    ax.set_ylim(min(b[1] for b in boxes) - 0.8, max(b[1] for b in boxes) + 0.8)
    ax.axis('off')
    plt.show()

In [ ]:
# the map of the laboratory: a random process, one record, two descriptions, one estimate
boxes = [
    (1.9, 4.1, 3.3, 'Random process\nall its records, task 1', 'BLUE'),
    (6.5, 4.1, 2.8, 'One realisation $x[n]$\nthe record we have', 'BLUE'),
    (10.6, 4.9, 4.0, 'In time: $r_{xx}[k]$\nhow samples are linked, task 3', 'SAGE'),
    (10.6, 3.0, 4.0, 'In frequency: $S_{xx}(f)$\nwhere the power is, task 4', 'BRICK'),
    (1.1, 0.6, 1.9, 'Record\n$N$ samples', 'AMBER'),
    (3.4, 0.6, 1.9, 'Segments\nof $D$ samples', 'AMBER'),
    (5.7, 0.6, 1.9, 'Window\n$H[n]$', 'AMBER'),
    (8.0, 0.6, 1.9, 'Squared FFT\nof each segment', 'AMBER'),
    (10.3, 0.6, 1.9, 'Average over\nthe segments', 'AMBER'),
    (12.6, 0.6, 1.9, 'Estimate\n$\\hat{S}(f)$', 'AMBER'),
]
arrows = [
    (0, 1, 'to', 'we get one'),
    (1, 2, 'to', ''),
    (1, 3, 'to', ''),
    (2, 3, 'both', 'Wiener–Khinchin, task 5'),
    (4, 5, 'to', ''), (5, 6, 'to', ''), (6, 7, 'to', ''), (7, 8, 'to', ''), (8, 9, 'to', ''),
    (9, 3, 'dashed', 'estimates'),
]
draw_flow(boxes, arrows, size=(12.5, 4.2))

## Task 1 · A random process and its realisations

**What we do.** A random process gives a different record every time we draw it. Each record is a **realisation**, $x^{(i)}[n]$. A detector gives us one. Fix the step $n$ and look across the realisations: the values form a random variable with its own histogram. The **expectation** $\mathcal{E}\{x[n]\}$ is the average across the realisations at a fixed $n$. The **time average** is the average along one realisation.

The process is a constant mean $\mu$ plus Gaussian noise of standard deviation $\sigma$: $x^{(i)}[n] = \mu + w^{(i)}[n]$. The function `realisations` draws $R$ records of $N$ samples. The function `two_averages` computes the two averages. The first figure shows a few realisations on the same axes and the histogram at one fixed step. The second figure compares the spread of the two averages.

> **The symbols**
>
> - **$\mu$** — the mean of the process
> - **$\sigma$** — the standard deviation of the noise
> - **$w^{(i)}[n]$** — the noise in realisation $i$

In [ ]:
def realisations(R, N, mu, sigma, seed=2026):
    """R records of the process x[n] = mu + w[n], one record per row.

    R, N  : number of realisations and number of samples per record (pure numbers)
    mu    : mean of the process, in the units of x
    sigma : standard deviation of the noise w, in the units of x
    Returns an array of shape (R, N). Row i is the realisation x^(i)[n]."""
    # white Gaussian noise, drawn again for every realisation and every step
    rng = np.random.default_rng(seed)
    w = sigma * rng.normal(size=(R, N))
    # the process is the constant mean plus the noise
    x = mu + w
    return x


def two_averages(x):
    """The two averages of a set of records.

    x : array of shape (R, N), one realisation per row
    Returns (expectation, time_average). The expectation has N values, one for
    each step: the average across the realisations. The time average has R
    values, one for each realisation: the average along the record."""
    # across the realisations, at each step n
    expectation = x.mean(axis=0)
    # along each realisation
    time_average = x.mean(axis=1)
    averages = (expectation, time_average)
    return averages

In [ ]:
# change these
R = 2000          # number of realisations
N = 200           # samples in one record
mu = 5.0          # mean of the process
sigma = 1.0       # standard deviation of the noise
n_fixed = 10      # the step at which we look across the realisations
n_shown = 4       # realisations drawn in the first figure

# draw the realisations and compute the two averages
x = realisations(R, N, mu, sigma)
expectation, time_average = two_averages(x)
print(f"expectation at n = {n_fixed}: {expectation[n_fixed]:.3f}   at n = 150: {expectation[150]:.3f}")
print("time averages of realisations 1, 2, 3:", np.round(time_average[:3], 3))
print(f"scatter of the expectation over n: {expectation.std():.3f}   sigma/sqrt(R) = {sigma / np.sqrt(R):.3f}")
print(f"scatter of the time averages: {time_average.std():.3f}   sigma/sqrt(N) = {sigma / np.sqrt(N):.3f}")

# first figure: a few realisations on the same axes and the histogram across them at one step
colours = [P['BLUE'], P['SAGE'], P['AMBER'], P['PLUM']]
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [3, 1]}, sharey=True)
for i in range(n_shown):
    axa.plot(x[i], color=colours[i % 4], lw=0.7, label=f'realisation {i + 1}')
axa.plot(expectation, color=P['BRICK'], lw=1.6, label='expectation')
axa.axvline(n_fixed, color=P['GREY_MID'], lw=1.0, ls=':')
axa.set(xlabel='step $n$', ylabel='$x[n]$')
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=n_shown + 1, frameon=False, fontsize=9)
axb.hist(x[:, n_fixed], bins=40, density=True, color=P['AMBER_LIGHT'], edgecolor=P['AMBER'], orientation='horizontal')
v = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 200)
axb.plot(np.exp(-0.5 * ((v - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi)), v, color=P['BRICK'], lw=1.4)
axb.set(xlabel=f'density at $n = {n_fixed}$', ylim=(mu - 4 * sigma, mu + 4 * sigma))
plt.show()

# second figure: the values of the two averages on the same axes
fig, ax = plt.subplots(figsize=(9, 3.4))
bins = np.linspace(mu - 4 * sigma / np.sqrt(N), mu + 4 * sigma / np.sqrt(N), 50)
ax.hist(time_average, bins=bins, density=True, histtype='step', color=P['BLUE'], lw=1.8, label=f'time averages, $R = {R}$ values')
ax.hist(expectation, bins=bins, density=True, histtype='step', color=P['BRICK'], lw=1.8, label=f'expectation, $N = {N}$ values')
ax.axvline(mu, color=P['GREY_MID'], lw=1.0, ls=':')
ax.set(xlabel='value of the average', ylabel='density')
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
plt.show()

**What to change.** First $R$: 200, 2000, 20000. Then $N$: 20, 200, 2000. Then $n_{\text{fixed}}$: the histogram must not depend on the step. Look at the histogram at one step and at the two curves of the second figure: which one gets narrower when $R$ grows and which when $N$ grows?

**Question.** What happens to the scatter of the expectation and to the scatter of the time averages? When can one record tell us the expectation?

<details>
<summary>Solution</summary>

With $R = 2000$ the expectation scatters by $0.022$ around $\mu = 5$ from one step to the next. This is $\sigma/\sqrt{R}$. With $N = 200$ the time averages scatter by $0.071$ from one realisation to the next. This is $\sigma/\sqrt{N}$. Raising $R$ to $20000$ narrows the expectation to $0.0076$ and leaves the time averages at $0.071$. Raising $N$ to $2000$ narrows the time averages to $0.022$ and leaves the expectation at $0.022$. The histogram at one step looks the same at every step: the process does not change with $n$.

The expectation is an average across realisations, which a detector does not have. Here the process does not change with $n$ and each realisation explores all its values. So the time average of one long record converges to the expectation. This is a stationary and ergodic process. It is the reason one record from a detector can measure the averages of its noise. A short record is not enough: with $N = 20$ the time averages scatter by $0.22$.

</details>


## Task 2 · Correlation between two random variables

**What we do.** Two random variables are **correlated** when they tend to move together: knowing one says something about the other. For two variables of zero mean and standard deviation 1 the **correlation coefficient** is the average of their product,

$$
\rho = \mathcal{E}\{x\,y\} ,
\qquad -1 \le \rho \le 1 .
$$

The function `correlated_pair` draws $M$ pairs with a chosen $\rho$: $y = \rho\,x + \sqrt{1 - \rho^{2}}\,z$, with $x$ and $z$ independent. The cell draws the pairs of three values of $\rho$ as clouds of points and prints $\rho$ estimated as the average of the product next to the value chosen.

> **The symbols**
>
> - **$\rho$** — the correlation coefficient
> - **$M$** — the number of pairs
> - **$z$** — a third Gaussian variable, independent of $x$

In [ ]:
def correlated_pair(M, rho, seed=7):
    """M pairs (x, y) of zero mean and unit variance with correlation rho.

    M    : number of pairs (pure number)
    rho  : correlation coefficient between -1 and 1 (pure number)
    Returns the tuple (x, y) of two arrays of M values."""
    # two independent Gaussian variables
    rng = np.random.default_rng(seed)
    x, z = rng.normal(size=(2, M))
    # y keeps the fraction rho of x and takes the rest from z, so its variance stays 1
    y = rho * x + np.sqrt(1 - rho ** 2) * z
    pair = (x, y)
    return pair

In [ ]:
# change these
rho_values = (0.0, 0.6, -0.9)     # the three correlation coefficients
M = 2000                          # number of pairs

# one cloud of points for each rho, with the average of the product
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), sharey=True)
for rho, ax in zip(rho_values, axes):
    x, y = correlated_pair(M, rho)
    rho_hat = np.mean(x * y)
    print(f"rho chosen = {rho:5.2f}   average of x y = {rho_hat:6.3f}   1/sqrt(M) = {1 / np.sqrt(M):.3f}")
    ax.plot(x, y, '.', color=P['BLUE'], ms=2, alpha=0.5)
    ax.set(xlabel='$x$', title=rf'$\rho = {rho:g}$', aspect='equal', xlim=(-4, 4), ylim=(-4, 4))
axes[0].set_ylabel('$y$')
plt.show()

**What to change.** First look at the three clouds: which way does each one lean? Then set one of the values of $\rho$ to $0.05$ and change $M$: 100, 2000, 100000. Compare the difference between the average of the product and $\rho$ with $1/\sqrt{M}$.

**Question.** From how many pairs can you tell $0.05$ from $0$? What does the sign of $\rho$ mean?

<details>
<summary>Solution</summary>

The averages of the product are $0.007$, $0.588$ and $-0.871$ for the chosen $0$, $0.6$ and $-0.9$. They differ from the chosen values by a few hundredths. This is the scatter of an average of $M = 2000$ products, of order $1/\sqrt{M} = 0.022$.

For $\rho = 0.05$ the averages are $0.068$ with $M = 100$, $0.056$ with $M = 2000$ and $0.053$ with $M = 100000$. At $M = 100$ the scatter is $0.1$, larger than $\rho$: we cannot tell $0.05$ from $0$. At $M = 2000$ the scatter is $0.022$ and the average is about two and a half times larger: it is barely seen. At $M = 100000$ the scatter is $0.003$ and the correlation is clear. A weak correlation needs many samples.

A positive $\rho$ means that the two variables tend to have the same sign. A negative $\rho$ means the opposite sign. For Gaussian variables $\rho = 0$ means independent. In general it means only that the product averages to zero.

</details>


## Task 3 · The autocorrelation: how a sample is linked to the next ones

**What we do.** The **autocorrelation** of a stationary process asks the question of task 2 about the samples of one record: how is $x[n]$ linked to the sample $k$ steps later?

$$
r_{xx}[k] = \mathcal{E}\{x[n]\,x[n+k]\} .
$$

For an ergodic process we estimate it along one record, as the average of $x[n]\,x[n+k]$ over $n$. We compare two noises. **White** noise $w[n]$ has no memory: $r_{ww}[k] = \sigma_{w}^{2}\,\delta[k]$, a single spike at $k = 0$. **Coloured** noise has memory. We make it from white noise with

$$
x[n] = a\,x[n-1] + w[n],
\qquad |a| < 1 ,
\qquad
r_{xx}[k] = \frac{\sigma_{w}^{2}\,a^{|k|}}{1 - a^{2}} .
$$

The function `coloured_noise` makes $N$ samples of it. The function `acf` estimates $r_{xx}[k]$ along one record. The first panel shows the white record and one coloured record on the same axes. The second panel shows the autocorrelation of the white noise and of several values of $a$ on the same axes, divided by its value at $k = 0$ so that the memories can be compared. The lines are the formula, $a^{|k|}$ after the division.

> **The symbols**
>
> - **$w[n]$** — white Gaussian noise of standard deviation $\sigma_{w}$
> - **$a$** — the memory of the coloured noise: how much of the last sample is kept
> - **$\delta[k]$** — 1 at $k = 0$ and 0 elsewhere

In [ ]:
def coloured_noise(N, a, sigma_w, seed=9):
    """N samples of the coloured noise x[n] = a x[n-1] + w[n].

    N       : number of samples (pure number)
    a       : memory, how much of the last sample is kept, |a| < 1 (pure number)
    sigma_w : standard deviation of the white noise w, in the units of x
    seed    : seed of the white noise; the same seed gives the same w for every a
    Returns an array of N samples, started in its stationary state."""
    # the white noise that drives the process
    w = sigma_w * np.random.default_rng(seed).normal(size=N)
    # the first sample already has the variance of the stationary process
    x = np.empty(N)
    x[0] = w[0] / np.sqrt(1 - a ** 2)
    # every new sample keeps the fraction a of the last one and adds new noise
    for n in range(1, N):
        x[n] = a * x[n - 1] + w[n]
    return x


def acf(x, k_max):
    """The autocorrelation r_xx[k] estimated along one record.

    x     : one record, in the units of x
    k_max : the largest lag, in steps
    Returns r_xx[k] for k = 0 ... k_max, in the units of x squared."""
    # for each lag k, the average over n of x[n] x[n+k]
    r = np.array([np.mean(x[:x.size - k] * x[k:]) for k in range(k_max + 1)])
    return r

In [ ]:
# change these
a_values = (0.5, 0.9, 0.99)     # the memories compared in the second panel
a_record = 0.9                  # the memory of the coloured record drawn in the first panel
sigma_w = 1.0                   # standard deviation of the white noise
N = 500000                      # samples in the record
k_max = 60                      # largest lag

# the white noise, and the autocorrelation of the white noise
w = sigma_w * np.random.default_rng(9).normal(size=N)
lags = np.arange(k_max + 1)
r_white = acf(w, k_max)

# the coloured records and their autocorrelations, with the formula to compare
colours = [P['BLUE'], P['AMBER'], P['BRICK'], P['TEAL']]
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4))
axa.plot(w[:200], color=P['GREY_MID'], lw=0.8, label='white')
axa.plot(coloured_noise(200, a_record, sigma_w), color=P['BLUE'], lw=1.2, label=f'coloured, $a = {a_record:g}$')
axa.set(xlabel='step $n$', ylabel='$x[n]$')
axa.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
axb.plot(lags, r_white / r_white[0], 'o', color=P['GREY_MID'], ms=4, label='white')
for a, colour in zip(a_values, colours):
    x = coloured_noise(N, a, sigma_w)
    r_formula = sigma_w ** 2 * a ** lags / (1 - a ** 2)
    print(f"a = {a:5.2f}: r_xx[0] = {acf(x, 0)[0]:6.3f}   formula {r_formula[0]:6.3f}   "
          f"steps to fall by e: {-1 / np.log(abs(a)):5.1f}")
    r_estimate = acf(x, k_max)
    axb.plot(lags, r_estimate / r_estimate[0], 'o', color=colour, ms=4, label=f'$a = {a:g}$')
    axb.plot(lags, r_formula / r_formula[0], color=colour, lw=1.0)
axb.set(xlabel='lag $k$', ylabel='$r_{xx}[k]\\,/\\,r_{xx}[0]$')
axb.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, frameon=False)
plt.show()

**What to change.** Change $a$ in `a_values`: first $(0.5, 0.9, 0.99)$ and then $(0.5, -0.8)$. Change `a_record` to $0.99$ and to $-0.8$ and look at the record. In each case ask how many steps the autocorrelation needs to fall and whether the points follow the lines.

**Question.** How far does the memory reach? What does a negative $a$ do to the record and to $r_{xx}[k]$?

<details>
<summary>Solution</summary>

For $a = 0.9$, $r_{xx}[0] = 5.27$, which is $\sigma_{w}^{2}/(1 - a^{2}) = 5.26$. The autocorrelation falls by a factor $\mathrm{e}$ in $-1/\ln a = 9.5$ steps. For $a = 0.5$ it falls in $1.4$ steps and the record looks almost white. For $a = 0.99$ it takes $99$ steps. The record wanders far and slowly and its variance, $50.2$, is 50 times that of the white noise. The memory reaches over about $-1/\ln|a|$ steps. In the second panel the points lie on the lines: one long record gives the autocorrelation.

For $a = -0.8$ each sample keeps the last one with the opposite sign. The record jumps up and down from step to step. The autocorrelation alternates in sign: $r_{xx}[k]/r_{xx}[0] = -0.80,\ 0.64,\ -0.51$ at $k = 1, 2, 3$, which is $(-0.8)^{k}$. Neighbours are anticorrelated.

</details>


## Task 4 · The power spectral density

**What we do.** The **power spectral density** says how the power of the process is shared among the frequencies. It is the expectation of the squared transform of a record, divided by its length:

$$
S_{xx}(f) = \lim_{N\to\infty}\,\mathcal{E}\left\{\frac{1}{N}\left|\sum_{n} x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n}\right|^{2}\right\} .
$$

We cannot take the limit and we cannot draw all the realisations of a detector. Here we can, so we follow the definition. The function `psd_average` takes $R$ records and computes $\frac{1}{N}|\sum_{n} x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n}|^{2}$ for each one with `np.fft.rfft`. Then it averages them. We compare with the formulas. White noise is flat, $S_{ww}(f) = \sigma_{w}^{2}$. Coloured noise is white noise through a filter,

$$
S_{xx}(f) = \frac{\sigma_{w}^{2}}{\left|1 - a\,\mathrm{e}^{-\mathrm{i} 2\pi f}\right|^{2}} .
$$

The figure shows the white noise and several values of $a$ on the same axes. The dashed curves are the formula.

**Uses** `coloured_noise` from task 3.

> **The symbols**
>
> - **$S_{xx}(f)$** — the power spectral density, power per unit of frequency

In [ ]:
def coloured_records(R, N, a, sigma_w):
    """R independent records of the coloured noise of task 3, one per row.

    R, N    : number of records and of samples per record (pure numbers)
    a       : memory of the coloured noise (pure number)
    sigma_w : standard deviation of the white noise, in the units of x
    Returns an array of shape (R, N)."""
    # one record for each seed, so that the records are independent
    records = np.array([coloured_noise(N, a, sigma_w, seed=100 + i) for i in range(R)])
    return records


def psd_average(records):
    """The average over the records of |sum_n x[n] exp(-i 2 pi f n)|^2 / N.

    records : array of shape (R, N), one record per row, in the units of x
    Returns S at f = 0, 1/N, ... up to 1/2, in the units of x squared
    per cycle per sample."""
    N = records.shape[1]
    # the squared transform of each record, divided by its length
    power = np.abs(np.fft.rfft(records, axis=1)) ** 2 / N
    # the expectation: the average over the records
    S = power.mean(axis=0)
    return S


def coloured_psd(f, a, sigma_w):
    """The PSD of the coloured noise, S = sigma_w^2 / |1 - a exp(-i 2 pi f)|^2.

    f       : frequencies in cycles per sample
    a       : memory (pure number)
    sigma_w : standard deviation of the white noise, in the units of x
    Returns S at each f, in the units of x squared per cycle per sample."""
    # the transfer function of the filter that makes the noise coloured
    transfer = 1 / (1 - a * np.exp(-2j * np.pi * f))
    # the white level sigma_w^2 times its squared modulus
    S = sigma_w ** 2 * np.abs(transfer) ** 2
    return S

In [ ]:
# change these
a_values = (0.5, 0.9)      # the memories of the coloured noise
sigma_w = 1.0              # standard deviation of the white noise
N = 512                    # samples in one record
R = 400                    # number of records in the average

# the white noise: flat at sigma_w^2
f = np.fft.rfftfreq(N)
white = sigma_w * np.random.default_rng(11).normal(size=(R, N))
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(f, psd_average(white), color=P['GREY_MID'], lw=1.0, label='white')
ax.axhline(sigma_w ** 2, color=P['GREY_MID'], lw=1.0, ls='--')

# each coloured noise: the average of the records on the formula
colours = [P['BLUE'], P['AMBER'], P['BRICK'], P['TEAL']]
for a, colour in zip(a_values, colours):
    S_estimate = psd_average(coloured_records(R, N, a, sigma_w))
    S_formula = coloured_psd(f, a, sigma_w)
    print(f"a = {a:5.2f}: S at f = 0: {S_estimate[0]:7.2f}   formula {S_formula[0]:7.2f}   "
          f"S at f = 1/2: {S_estimate[-1]:6.3f}   formula {S_formula[-1]:6.3f}")
    ax.semilogy(f, S_estimate, color=colour, lw=1.0, label=f'$a = {a:g}$, average')
    ax.semilogy(f, S_formula, color=colour, lw=1.6, ls='--')
ax.set(xlabel='frequency $f$ (cycles per sample)', ylabel='$S_{xx}(f)$', xlim=(0, 0.5))
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=4, frameon=False)
plt.show()

**What to change.** First look at the white noise: how high is the line? Change `a_values` to $(0.5, 0.9, -0.9)$. Double $\sigma_{w}$. Then try $a = 0.99$ with $N = 4096$. For each case ask where the power is and how the level and the shape change.

**Question.** Where does the power go? What does $\sigma_{w}$ change?

<details>
<summary>Solution</summary>

The white noise is flat at $\sigma_{w}^{2} = 1$: the average over the records is $1.00$. For $a = 0.9$ the density is $98$ at $f = 0$. The formula, $\sigma_{w}^{2}/(1 - a)^{2}$, gives $100$. At $f = \tfrac{1}{2}$ it is $0.265$ and the formula, $\sigma_{w}^{2}/(1 + a)^{2}$, gives $0.277$. The small differences are the scatter of an average of 400 records. For $a = 0.5$ the slope is gentle: $3.98$ at $f = 0$ and $0.418$ at $\tfrac{1}{2}$. The formulas give $4$ and $0.444$.

For $a = 0.99$ almost all the power is in a narrow band near $f = 0$, where the formula gives $10^{4}$. With $N = 512$ the average gives only $7980$ and with $N = 4096$ it gives $8970$. The record must be much longer than the memory of $99$ steps to show the peak. For $a = -0.9$ the picture is reversed: $0.28$ at $f = 0$ and $91.6$ at $f = \tfrac{1}{2}$. The power sits at the fastest oscillation the samples can show, as the jumping record of task 3 said.

Doubling $\sigma_{w}$ multiplies the density by four at every frequency: for $a = 0.9$ it is $392$ at $f = 0$ and $1.06$ at $f = \tfrac{1}{2}$. It changes the level and not the shape. The shape is set by the memory $a$.

</details>


## Task 5 · The link: the spectrum is the transform of the autocorrelation

**What we do.** The **Wiener–Khinchin theorem** says that tasks 3 and 4 measured the same thing twice:

$$
S_{xx}(f) = \sum_{k=-\infty}^{\infty} r_{xx}[k]\,\mathrm{e}^{-\mathrm{i} 2\pi f k} ,
\qquad
r_{xx}[0] = \int_{-1/2}^{1/2} S_{xx}(f)\,\mathrm{d}f .
$$

The second line says that the area under the density is the variance: the total power. There are two routes to $S_{xx}(f)$. The first goes through the autocorrelation of one record. The second averages the squared transforms of many records, as in task 4. The drawing shows both.

The function `psd_from_acf` sums the transform of the autocorrelation up to the lag $k_{\max}$. Since $r_{xx}[-k] = r_{xx}[k]$ the sum is $r_{xx}[0] + 2\sum_{k \ge 1} r_{xx}[k]\cos 2\pi f k$. The figure draws the two routes and the formula on the same axes. The cell prints the area under the density next to $r_{xx}[0]$.

**Uses** `coloured_noise` and `acf` from task 3; `coloured_records`, `psd_average` and `coloured_psd` from task 4; `draw_flow` from the opening.

> **The symbols**
>
> - **$k_{\max}$** — the largest lag kept in the sum

In [ ]:
# the two routes to the same spectrum
boxes = [
    (1.3, 1.6, 1.9, 'Records\nof the process', 'BLUE'),
    (4.4, 2.7, 3.0, 'Autocorrelation\n$r_{xx}[k]$, task 3', 'SAGE'),
    (8.2, 2.7, 3.0, 'Transform of\n$r_{xx}[k]$', 'SAGE'),
    (4.4, 0.5, 3.0, 'Squared transform\nof each record', 'AMBER'),
    (8.2, 0.5, 3.0, 'Average over\nthe records, task 4', 'AMBER'),
    (11.7, 1.6, 2.2, '$S_{xx}(f)$\nthe same curve', 'BRICK'),
]
arrows = [(0, 1, 'to', ''), (1, 2, 'to', ''), (0, 3, 'to', ''), (3, 4, 'to', ''), (2, 5, 'to', ''), (4, 5, 'to', '')]
draw_flow(boxes, arrows, size=(11, 3.4))

In [ ]:
def psd_from_acf(r, f):
    """The transform of the autocorrelation, S(f) = r[0] + 2 sum_k r[k] cos(2 pi f k).

    r : autocorrelation r_xx[k] for k = 0 ... k_max, in the units of x squared
    f : frequencies in cycles per sample
    Returns S at each f, in the units of x squared per cycle per sample."""
    # the lags k = 1 ... k_max; the lags -k give the same cosine, hence the factor 2
    k = np.arange(1, r.size)
    # the sum over the lags, for every frequency at once
    S = r[0] + 2 * np.sum(r[1:, None] * np.cos(2 * np.pi * np.outer(k, f)), axis=0)
    return S

In [ ]:
# change these
a = 0.9              # memory of the coloured noise
sigma_w = 1.0        # standard deviation of the white noise
k_max = 100          # largest lag kept in the transform

# route 1: the autocorrelation along one long record, then its transform
x = coloured_noise(200000, a, sigma_w)
r = acf(x, k_max)
f = np.linspace(0, 0.5, 501)
S_acf = psd_from_acf(r, f)
# route 2: the average of the squared transforms of 400 records, as in task 4
S_average = psd_average(coloured_records(400, 512, a, sigma_w))
# the formula of task 4
S_formula = coloured_psd(f, a, sigma_w)
# the area under the density from -1/2 to 1/2 is twice the area from 0 to 1/2
area = 2 * np.trapezoid(S_acf, f)
print(f"area under S = {area:.3f}   r_xx[0] = {r[0]:.3f}   variance of the record = {x.var():.3f}")

fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 3.8))
axa.plot(np.arange(k_max + 1), r, color=P['SAGE'], lw=1.4)
axa.set(xlabel='lag $k$', ylabel='$r_{xx}[k]$, along one record')
axb.semilogy(np.fft.rfftfreq(512), S_average, color=P['AMBER'], lw=0.9, label='average of the records')
axb.semilogy(f, S_acf, color=P['SAGE'], lw=1.6, label='transform of $r_{xx}$')
axb.semilogy(f, S_formula, color=P['BRICK'], lw=1.4, ls='--', label='formula')
axb.set(xlabel='frequency $f$ (cycles per sample)', ylabel='$S_{xx}(f)$', xlim=(0, 0.5))
axb.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=3, frameon=False)
plt.show()

**What to change.** Change $a$ to $0.5$ and to $0.99$ and compare the width of the autocorrelation with the width of the density. Then set $k_{\max} = 5$ with $a = 0.99$. Then $k_{\max} = 400$ with $a = 0.99$.

**Question.** When the autocorrelation gets wider what happens to the density? What goes wrong when $k_{\max}$ is too small and why?

<details>
<summary>Solution</summary>

For $a = 0.9$ the area under the density is $5.235$. It is equal to $r_{xx}[0] = 5.235$ and to the variance of the record, $5.234$. For $a = 0.5$ the area is $1.333$. The two descriptions agree.

For $a = 0.99$ the autocorrelation lasts about a hundred steps and the density is a narrow peak at $f = 0$. For $a = 0.5$ the autocorrelation lasts one or two steps and the density is broad. A wide autocorrelation gives a narrow spectrum and a short one gives a wide spectrum. They are a Fourier pair, like a long ringing and a narrow resonance in `H02`.

With $k_{\max} = 5$ and $a = 0.99$ the sum keeps only the first steps of a memory that lasts a hundred. It misses the slow part of the process: the peak at $f = 0$ comes out $510$ instead of $10^{4}$ and the curve goes below zero at some frequencies, which a power cannot do. With $k_{\max} = 100$ the peak is $5890$. With $k_{\max} = 400$ it is $8540$ and the curve stays positive. To transform the autocorrelation we must keep all the lags where it is not yet zero.

</details>


## Task 6 · Windowing: a weak line beside a strong one

**What we do.** A detector record holds lines: a sinusoid of the mains or a violin mode. To estimate the spectrum we take $N$ samples and transform them. Cutting $N$ samples out of a longer signal multiplies it by a rectangle. In frequency the rectangle spreads each line over the whole band, in sidelobes that fall slowly. A strong line leaks over a weak one and hides it. A **window** $H[n]$ that brings the record smoothly to zero at its ends lowers the sidelobes. We use the **Hann** window, $H[n] = \sin^{2}(\pi n/N)$. We divide by its mean square $U$ so that the level stays right:

$$
\hat{S}(f) = \frac{1}{N\,U}\left|\sum_{n=0}^{N-1} H[n]\,x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n}\right|^{2},
\qquad
U = \frac{1}{N}\sum_{n=0}^{N-1} H^{2}[n] .
$$

The record holds a strong line of amplitude 1 at $f_{1}$, a weak line of amplitude $A_{2}$ at $f_{2}$ and a little white noise. The function `two_lines` makes it. The function `windowed_spectrum` computes $\hat{S}$ with the rectangle or with Hann. The frequencies are chosen between two frequencies of the transform, the worst case for leakage. The figure draws the two spectra on the same axes.

> **The symbols**
>
> - **$H[n]$** — the window: 1 at every sample for the rectangle and $\sin^{2}(\pi n/N)$ for Hann
> - **$U$** — the mean square of the window
> - **$f_{1}$, $f_{2}$** — the frequencies of the strong and of the weak line, in cycles per sample
> - **$A_{2}$** — the amplitude of the weak line

In [ ]:
def two_lines(N, f1, f2, A2, noise, seed=3):
    """A strong line of amplitude 1, a weak line of amplitude A2 and white noise.

    N      : number of samples (pure number)
    f1, f2 : frequencies of the strong and of the weak line, in cycles per sample
    A2     : amplitude of the weak line, in units of the strong one
    noise  : standard deviation of the white noise, in the same units
    Returns the record x[n], n = 0 ... N-1."""
    n = np.arange(N)
    # the two sinusoids
    strong = np.sin(2 * np.pi * f1 * n)
    weak = A2 * np.sin(2 * np.pi * f2 * n)
    # the white noise that fills the floor
    floor = noise * np.random.default_rng(seed).normal(size=N)
    x = strong + weak + floor
    return x


def windowed_spectrum(x, window):
    """The spectral estimate S(f) of one stretch of data, with a window.

    x      : the stretch, N samples in the units of x
    window : 'rect' for the rectangle or 'hann' for the Hann window
    Returns S at f = 0, 1/N, ... up to 1/2, in the units of x squared
    per cycle per sample."""
    N = x.size
    # the window: 1 everywhere for the rectangle, sin^2(pi n / N) for Hann
    H = np.ones(N) if window == 'rect' else np.sin(np.pi * np.arange(N) / N) ** 2
    # the mean square of the window puts back the power it removes
    U = np.mean(H ** 2)
    # the squared transform of the windowed data
    S = np.abs(np.fft.rfft(H * x)) ** 2 / (N * U)
    return S

In [ ]:
# change these
N = 1024                  # samples in the record
f1 = 100.5 / 1024         # frequency of the strong line, cycles per sample
f2 = 140.5 / 1024         # frequency of the weak line, cycles per sample
A2 = 1e-3                 # amplitude of the weak line
noise = 1e-5              # standard deviation of the white noise

# the record and its two spectra on the same axes
x = two_lines(N, f1, f2, A2, noise)
f = np.fft.rfftfreq(N)
fig, ax = plt.subplots(figsize=(9, 4))
for window, colour in (('rect', P['GREY_MID']), ('hann', P['BLUE'])):
    S = windowed_spectrum(x, window)
    near = np.abs(f - f2) < 3 / N
    print(f"{window}: S near the weak line {S[near].max():.2e}   S halfway between the lines"
          f" {S[np.argmin(np.abs(f - (f1 + f2) / 2))]:.2e}")
    ax.semilogy(f, S, color=colour, lw=1.0, label=window)
ax.axvline(f2, color=P['BRICK'], lw=0.8, ls=':')
ax.set(xlabel='frequency $f$ (cycles per sample)', ylabel=r'$\hat{S}(f)$', xlim=(0, 0.5), ylim=(1e-14, 1e3))
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
plt.show()

**What to change.** First $A_{2}$: $10^{-1}$, $10^{-3}$ and $10^{-5}$. Then bring the weak line close to the strong one, $f_{2} = 102.5/1024$. The dotted line marks $f_{2}$.

**Question.** When do you need the window? What does the window cost?

<details>
<summary>Solution</summary>

With $A_{2} = 10^{-3}$ the rectangle shows at $f_{2}$ the leakage of the strong line, $1.4\times10^{-2}$. It is a hundred times the weak line and hides it. Hann shows the weak line, $1.2\times10^{-4}$. Halfway between the two lines Hann gives $3\times10^{-7}$ and the rectangle $6\times10^{-2}$. Far from the lines Hann reaches the noise floor, $6\times10^{-11}$. The rectangle stays at $1.2\times10^{-4}$.

With $A_{2} = 10^{-1}$ both windows show the weak line, $1.27$ with the rectangle and $1.23$ with Hann: it is stronger than the leakage. With $A_{2} = 10^{-5}$ only Hann shows it, $3\times10^{-8}$. It is above the leakage of the strong line, $4\times10^{-9}$. It is also above the noise.

The window is needed when the lines differ by more than the sidelobes of the rectangle fall, a few orders of magnitude. Its cost is resolution. The main lobe of Hann is twice as wide as that of the rectangle. With $f_{2} = 102.5/1024$ the weak line is two frequency steps from the strong one. It sits on the main lobe of the strong line, $5$ with Hann and $11$ with the rectangle, a million times above the weak line. No window shows it. Lines this close need a longer record, not a better window.

</details>


## Task 7 · Bias and variance: from one periodogram to Welch

**What we do.** The spectrum of tasks 4 and 5 was an average over many records. A detector gives one record of $N$ samples. Its **periodogram**, $\hat{S}(f) = \frac{1}{N}|\sum_{n} x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n}|^{2}$, is the squared transform of task 6 with the rectangle. It has two errors. The **bias** is the mean of the estimate minus the true $S_{xx}(f)$. The **variance** is the spread of the estimate around its mean. The periodogram has a small bias. Its standard deviation is as large as its mean and a longer record does not lower it.

**Welch's method** lowers the variance. We cut the record into segments of $D$ samples, multiply each by the window $H[n]$, take the squared transform of each and average the $K$ results. The segments may overlap. The drawing shows the steps. The variance falls as $1/K$. But the segment is shorter than the record, so the estimate smooths the spectrum over about $1/D$ and the bias grows. For a fixed record we must choose a compromise.

The function `welch_estimate` cuts the record, windows each segment with `windowed_spectrum` and averages. The function `compare_with_formula` measures the two errors against the true spectrum of the coloured noise. The figure draws the periodogram and the Welch estimates for several $D$ on the same axes. The right panel enlarges the peak at low frequency, where the bias shows.

You can skip this task: tasks 1 to 6 are complete without it.

**Uses** `coloured_noise` from task 3; `coloured_psd` from task 4; `windowed_spectrum` from task 6; `draw_flow` from the opening.

> **The symbols**
>
> - **$K$** — the number of segments
> - **$\hat{S}_{W}(f)$** — the Welch estimate: the average of the $K$ windowed periodograms

In [ ]:
# Welch's method: from one record to one estimate
boxes = [
    (1.0, 1.0, 1.7, 'Record\n$N$ samples', 'AMBER'),
    (3.5, 1.0, 2.3, '$K$ segments\nof $D$ samples,\nwith overlap', 'AMBER'),
    (6.15, 1.0, 2.0, 'Multiply each\nby $H[n]$', 'AMBER'),
    (8.75, 1.0, 2.2, 'Squared FFT\nof each segment', 'AMBER'),
    (11.3, 1.0, 1.9, 'Average\nover $K$', 'AMBER'),
    (13.6, 1.0, 1.7, '$\\hat{S}_{W}(f)$', 'BRICK'),
]
arrows = [(0, 1, 'to', ''), (1, 2, 'to', ''), (2, 3, 'to', ''), (3, 4, 'to', ''), (4, 5, 'to', '')]
draw_flow(boxes, arrows, size=(12.5, 1.9))

In [ ]:
def welch_estimate(x, D, overlap):
    """The Welch estimate of S(f): the average of the windowed periodograms of the segments.

    x       : the record, N samples in the units of x
    D       : samples in one segment (pure number)
    overlap : fraction of a segment shared with the next one, from 0 to less than 1
    Returns (f, S, K): the frequencies in cycles per sample, the estimate in
    the units of x squared per cycle per sample and the number of segments."""
    # the segments start every D (1 - overlap) samples
    step = int(D * (1 - overlap))
    starts = np.arange(0, x.size - D + 1, step)
    # the Hann periodogram of each segment, from task 6
    periodograms = np.array([windowed_spectrum(x[s:s + D], 'hann') for s in starts])
    # the average over the segments
    S = periodograms.mean(axis=0)
    f = np.fft.rfftfreq(D)
    estimate = (f, S, starts.size)
    return estimate


def compare_with_formula(f, S, a, sigma_w):
    """The two errors of an estimate, against the true spectrum of the coloured noise.

    f, S    : frequencies in cycles per sample and the estimate
    a       : memory of the coloured noise (pure number)
    sigma_w : standard deviation of the white noise, in the units of x
    Returns (bias_at_peak, scatter). bias_at_peak is the mean of the estimate
    divided by the mean of the truth over f < 0.006 where the peak is: 1 means no bias.
    scatter is the standard deviation of the estimate divided by the truth
    over 0.1 < f < 0.5, where the spectrum is smooth: it measures the variance."""
    # the estimate in units of the true spectrum
    truth = coloured_psd(f, a, sigma_w)
    ratio = S / truth
    # the bias at the peak
    near_peak = f < 0.006
    bias_at_peak = S[near_peak].mean() / truth[near_peak].mean()
    # the spread where the spectrum is smooth
    scatter = ratio[(f > 0.1) & (f < 0.5)].std()
    errors = (bias_at_peak, scatter)
    return errors

In [ ]:
# change these
a = 0.99                       # memory of the coloured noise
sigma_w = 1.0                  # standard deviation of the white noise
N = 2 ** 17                    # samples in the record
D_values = (64, 512, 4096)     # segment lengths compared
overlap = 0.5                  # fraction shared by neighbouring segments

# one record of coloured noise: this is all the data we have
x = coloured_noise(N, a, sigma_w)
f_all = np.fft.rfftfreq(N)

# the periodogram of the whole record and the true spectrum, in both panels
periodogram = windowed_spectrum(x, 'rect')
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.2))
for ax in (axa, axb):
    ax.semilogy(f_all[1:], periodogram[1:], color=P['GREY_LIGHT'], lw=0.6, label='periodogram, $D = N$')
    ax.semilogy(f_all[1:], coloured_psd(f_all[1:], a, sigma_w), color=P['INK'], lw=1.2, ls='--', label='true $S_{xx}$')

# the Welch estimate for each segment length, with its two errors
colours = [P['BLUE'], P['AMBER'], P['BRICK'], P['TEAL']]
for D, colour in zip(D_values, colours):
    f, S, K = welch_estimate(x, D, overlap)
    bias, scatter = compare_with_formula(f, S, a, sigma_w)
    print(f"D = {D:5d}: K = {K:4d}   estimate/truth at the peak: {bias:5.2f}   "
          f"scatter of estimate/truth: {scatter:5.3f}   1/sqrt(N/D) = {1 / np.sqrt(N / D):5.3f}")
    for ax in (axa, axb):
        ax.semilogy(f, S, color=colour, lw=1.4, label=f'Welch, $D = {D}$')
axa.set(xlabel='frequency $f$ (cycles per sample)', ylabel=r'$S_{xx}(f)$', xlim=(0, 0.5), ylim=(0.05, 3e4))
axb.set(xlabel='frequency $f$ (cycles per sample), the peak enlarged', xlim=(0, 0.01), ylim=(200, 2e4))
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False, fontsize=8.5)
plt.show()

**What to change.** Change `D_values` to $(64, 512, 4096)$ and then to $(32, 128, 1024)$ and look at the printed errors and at the peak in the right panel. Set `overlap` to $0$ and then to $0.75$. Make the record four times longer with $N = 2^{17}$.

**Question.** What happens to the bias and to the variance when $D$ gets shorter? Why can a longer record lower both?

<details>
<summary>Solution</summary>

The periodogram of the whole record is the grey curve: its scatter around the true spectrum is $1.0$, as large as the spectrum itself. Welch lowers it. With $N = 2^{17}$ and half overlap, $D = 64$ gives $K = 4095$ segments and a scatter of $0.012$. $D = 512$ gives $K = 511$ and $0.046$. $D = 4096$ gives $K = 63$ and $0.135$. Without overlap the scatter is $0.019$, $0.067$ and $0.18$, close to $1/\sqrt{N/D} = 0.022$, $0.062$ and $0.177$. Half overlap lowers it by about a third, because the Hann window gives little weight to the samples at the edges of a segment. Going from $0.5$ to $0.75$ gains almost nothing: the segments are no longer independent.

The bias is at the peak. The true spectrum is $10^{4}$ at $f = 0$ and the peak is about $0.0016$ wide. The estimate at the peak is $0.17$ of the truth for $D = 64$, $0.82$ for $D = 512$ and $0.91$ for $D = 4096$. A short segment smooths the spectrum over about $1/D$. With $D = 64$ that is $0.016$, ten times the width of the peak. The peak is lost. With $D = (32, 128, 1024)$ the peak is at $0.09$, $0.30$ and $0.88$.

A shorter $D$ lowers the variance and raises the bias. We cannot have both small with a fixed record. A longer record can. With $N = 2^{19}$ the scatter falls to $0.004$, $0.021$ and $0.064$ and the peak rises to $0.19$, $0.91$ and $1.00$. The choice of $D$ is set by the narrowest feature we must resolve.

</details>


## What to hand in

The answer to the question of each task, in two or three sentences, with the parameters you changed and what you saw: the scatter of the expectation and of the time average; when a correlation can be seen; how far the memory of the coloured noise reaches; where its power goes; why a wide autocorrelation gives a narrow spectrum; when a window is needed and what it costs; how the segment length trades bias against variance.

> **Where this goes next**
>
> You have seen a random process from its realisations, its autocorrelation and its spectrum. You have seen how a window lets a weak line be seen and how segments make an estimate steady. The noise of a detector is described with these same tools.

## Software

`numpy` throughout; `np.fft.rfft` gives the transforms of tasks 4 to 7. The window is described in (Harris 1978). The random processes follow (Therrien 1992) and (Papoulis and Pillai 2002) and the spectral estimates (Kay 1988).

**References**

- Therrien (1992). *Discrete Random Signals and Statistical Signal Processing*.
- Papoulis and Pillai (2002). *Probability, Random Variables and Stochastic Processes*.
- Kay (1988). *Spectral Estimation*.
- Harris (1978). *On the use of windows for harmonic analysis with the discrete Fourier transform*.